In [1]:
using LinearAlgebra
using BoundaryValueDiffEq
using Plots
using DifferentialEquations
using Polynomials

In [ ]:
# tspan = (0,1.0e1)
# initial_vec = [0,0,5103341351120374,0,-0.6151547026271073]
# function com_von_u_v!(du, u , p, t)
#     psi = u[1]
#     dpsi = u[2]
#     d2psi = u[3]
#     v = u[4]
#     dv = u[5]
#     du[1] = dpsi
#     du[2] = d2psi
#     d3psi = dpsi^2 - 2 * psi * d2psi - (v + 1.0e0)^2
#     du[3] = d3psi
#     du[4] = dv 
#     ddv = 2 * (v + 1) * dpsi - 2 * psi * dv                         
#     du[5] = ddv
# end
# function bc1!(residual, u , p , t )
#     residual[1] = u[begin][1] - 0
#     residual[2] = u[begin][2] - 0 
#     residual[3] = u[begin][4] - 0
#     residual[1] = u[end][2] - 0
#     residual[2] = u[end][4] + 1e0   
# end
# prob = BVProblem(com_von_u_v! , bc1!, initial_vec , tspan)
# sol1 = solve(prob, Shooting(Vern7()) , dt = 0.01)
# plot(sol1)

In [ ]:
tspan = (0,20)
initial_vec = [0,0,0,0,0]
function oneDiskODE!(du, u , p, t)
    U = u[1]
    dU = u[2]
    V = u[3]
    dV = u[4]
    W = u[5]
    du[1] = dU
    ddU = U^2 + W*dU - (V+1.0e0)^2
    du[2] = ddU
    ddV = 2.0e0*U*(V + 1.0e0) + W*dV
    du[3] = dV
    du[4] = ddV                          
    du[5] = -2.0e0*U
end
function oneDiskbc!(residual, u , p, t)

    residual[1] = u[begin][1] 
    residual[2] = u[begin][3] 
    residual[3] = u[begin][5] 
    residual[4] = u[end][1] 
    residual[5] = u[end][3] + 1.0e0
end
prob = BVProblem(oneDiskODE!, oneDiskbc!, [0.0, 0.5103341351120374, 0.0, -0.6151547026271073, 0.0] ,tspan, dtmax=0.01)
sol = solve(prob, Shooting(Vern7()), dt=0.001)

In [ ]:
t=range(0.0, 20, 20001)
u=sol(t)
plot(t, u[1, :], label="u")
plot!(t, u[3, :], label = "v")
plot!(t, u[5, :], label = "w")

In [ ]:
# U,V,W = zeros(20001,1)
U = u[1 , :]
dU = u[2 , :]
V = -1 * u[3 , :]
dV = -1 * u[4 , :]
W = -1 * u[5 , :]
phi = 0.001 .* cumsum(U)
M = fit(t , phi , 10) #psi
M1 = fit(t , U , 10)
M2 = fit(t , dU , 10)
M3 = fit(t , dV , 10)

In [ ]:
plot(t, U, label="u")
plot!(t, V, label = "v")
plot!(t, dU, label = "dU")
plot!(t, dV, label = "dV")
plot!(t, phi, label = "phi")
plot!(t, M.(t))
plot!(t, M1.(t))
plot!(t, M2.(t))
plot!(t, M3.(t))

In [ ]:
sigma = 0.7
tspan = (0,20)
#compute q
initial_vec = [0,0]
function com_von_q(du,u,p,t)
    q = u[1]
    dq = u[2]
    du[1] = u[2]
    du[2] = -2 * sigma * p(t) * u[2]
    
end
function bc1!(residual,u,p,t)
    residual[1] = u[begin][1] - 1
    residual[2] = u[end][1] 
    
end
N = t -> -0.008595159814265714 + 0.06257232655084231*t + 0.14386051336283018*t^2 - 0.08377768506114354*t^3 + 0.022082907193907726*t^4 - 0.0034028924250243896*t^5 + 0.00032947627288989947*t^6 - 2.0318308325353595e-5*t^7 + 7.751687800458372e-7*t^8 - 1.6682624336558755e-8*t^9 + 1.5488345724832982e-10*t^10
prob = BVProblem(com_von_q, bc1!, initial_vec,tspan, N)
sol = solve(prob, MIRK4(), dt=0.001)
t = range(0.0, 20, 20001)
u = sol(t)
q = u[1, :]
dq = u[2,:]

In [ ]:
#compute f
initial_vec = [0,0]
N = t -> -0.008595159814265714 + 0.06257232655084231*t + 0.14386051336283018*t^2 - 0.08377768506114354*t^3 + 0.022082907193907726*t^4 - 0.0034028924250243896*t^5 + 0.00032947627288989947*t^6 - 2.0318308325353595e-5*t^7 + 7.751687800458372e-7*t^8 - 1.6682624336558755e-8*t^9 + 1.5488345724832982e-10*t^10
N1 = t -> 0.008421938789645896 + 0.4553658705564695*t - 0.4199134291034149*t^2 + 0.16941940627324084*t^3 - 0.03893769561471858*t^4 + 0.005586870025576967*t^5 - 0.0005178926520379186*t^6 + 3.10180910802705e-5*t^7 - 1.1588823184837126e-6*t^8 + 2.455228500680004e-8*t^9 - 2.251828239294518e-10*t^10
N2 = t -> 0.5087667652449482 - 1.0072073248902742*t + 0.6784148506770165*t^2 - 0.23839839888347267*t^3 + 0.05047920459445594*t^4 - 0.00685041906698698*t^5 + 0.000609633420407717*t^6 - 3.53869378484485e-5*t^7 + 1.2896800697589372e-6*t^8 - 2.6777586928974654e-8*t^9 + 2.415219771143271e-10*t^10
N3 = t ->0.6383016840600214 - 0.17753053559377244*t - 0.14961867365209877*t^2 + 0.1064576265379749*t^3 - 0.03032486741680989*t^4 + 0.004901795356919967*t^5 - 0.0004909084795634821*t^6 + 3.105212695635389e-5*t^7 - 1.2083784942903816e-6*t^8 + 2.6421714859512894e-8*t^9 - 2.4850408634132963e-10*t^10
function com_von_f(du,u,p,t)
    f = u[1]
    df = u[2]
    du[1] = u[2]
    du[2] = -2 * sigma * N(t) * u[2] + 2 * sigma * N1(t) *u[1] + 2 * sigma * (N2(t)^2 + N3(t)^2) 
end
function bc2!(residual,u,p,t)
    residual[1] = u[begin][1] - 0
    residual[2] = u[end][1] - 0
end
prob = BVProblem(com_von_f, bc2!, initial_vec,tspan)
sol = solve(prob, MIRK4(), dt=0.001)

In [ ]:
t = range(0.0, 20, 20001)
u = sol(t)
f = u[1, :]
df = u[2,:]

In [ ]:
plot(t, f, label= "f" )
plot!(t, df, label = "df" , xlims=[0,10])

In [ ]:
plot(t, q, label="q")
plot!(t, dq, label = "dq",xlims=[0,10])

In [ ]:
function com_von_T(f ,q ,Mx ,gamma ,Tw)
    T = 1 .-(gamma-1)/2 * Mx^2 .* f .+ (Tw - 1) .*q
    return T
end

In [ ]:
Mx = 8
gamma = 1.4
Tw = 0
T = com_von_T(f , q , Mx ,gamma ,Tw)

In [ ]:
plot(t,T)